Juan Esteban Camacho Castro, Juan Diego Serpa Ruiz

# Semana 10, Transformación de fechas, texto y discretización

## Carga de librerías y dataset

In [1]:
import pandas as pd

In [4]:
df = pd.read_csv("S09_PD_Camacho_Serpa_DatasetCodificado.csv", low_memory=False)

## Conversión de los días de la semana

In [5]:
df["fecha_pedido"] = pd.to_datetime(df["fecha_pedido"])

In [6]:
df["fecha_pedido"].dtype

dtype('<M8[us]')

In [7]:
df["anio_pedido"] = df["fecha_pedido"].dt.year
df["mes_pedido"] = df["fecha_pedido"].dt.month
df["trimestre_pedido"] = df["fecha_pedido"].dt.quarter
df["dia_semana_pedido"] = df["fecha_pedido"].dt.day_name()
df["es_fin_semana_pedido"] = df["fecha_pedido"].dt.dayofweek >= 5

## Variables temporales extraídas

Se crearon 5 nuevas columnas indicando el año, el mes, el trimestre, el nombre del día y el día; llamadas respectivamente "anio_pedido", "mes_pedido", "trimestre_pedido", "dia_semana_pedido", "es_fin_semana_pedido".

In [8]:
df["comentario_cliente"] = df["comentario_cliente"].fillna("")

In [9]:
df["longitud_comentario"] = df["comentario_cliente"].str.len()

In [10]:
df["num_palabras_comentario"] = df["comentario_cliente"].str.split().str.len()

In [11]:
df["tiene_comentario"] = df["comentario_cliente"].str.len() > 0

In [12]:
df["menciona_demora"] = df["comentario_cliente"].str.contains(
    "tarde|demora|retraso|no llegó", case=False, na=False
)

In [13]:
df["menciona_demora"].value_counts()

menciona_demora
False    603303
Name: count, dtype: int64

## Variables de texto extraídas

La columna "longitud_comentario" muestra cuantos carácteres tienen los comentarios de los clientes; "num_palabras_comentarios" al igual que la anterior columna muestra la longitud de los comentarios, pero en términos de cuantas palabras tienen, sin espacios; "tiene_comentario" muestra si la columna no está vacía; "menciona_demora" recorre los comentarios buscando palabras alusivas a si los usuarios mencionan demora, si es así, las clasifica de manera booleana

## Separación en entrenamiento y prueba

In [14]:
from sklearn.model_selection import train_test_split

In [15]:
df_entrenamiento, df_prueba = train_test_split(df, test_size=0.2, random_state=42)

In [16]:
print("Filas entrenamiento:", len(df_entrenamiento))
print("Filas prueba:", len(df_prueba))

Filas entrenamiento: 482642
Filas prueba: 120661


## Discretización de edad_cliente

In [17]:
df["grupo_edad"] = pd.cut(df["edad_cliente"], bins=[17, 25, 35, 45, 60, 100], labels=["18 a 25", "26 a 35", "36 a 45", "46 a 60", "61 y más"])

In [18]:
df["grupo_edad"].value_counts().sort_index()

grupo_edad
18 a 25      78212
26 a 35      98109
36 a 45      97643
46 a 60     147503
61 y más    146809
Name: count, dtype: int64

Los bins son reglas fijas del negocio, entonces no hay riesgo de fuga de datos al aplicar la discretización sobre el dataset completo antes de dividir.

## Nueva división con grupo_edad incluida

Volvemos a dividir sobre el df que ya tiene la columna `grupo_edad`, usando el mismo `random_state=42` para que las mismas filas queden en cada conjunto.

In [19]:
df_entrenamiento, df_prueba = train_test_split(df, test_size=0.2, random_state=42)

## Sesgo de monto_compra y transformación logarítmica

In [21]:
df["monto_compra"].skew()

np.float64(1.019054942947812)

In [22]:
import numpy as np
df["monto_compra_log"] = np.log1p(df["monto_compra"])

In [23]:
df["monto_compra_log"].skew()

np.float64(-2.9633245462652957)

## Box-Cox

In [24]:
from sklearn.preprocessing import PowerTransformer

In [31]:
transformador_boxcox = PowerTransformer(method="box-cox")
transformador_boxcox.fit(df_entrenamiento[["monto_compra"]] + 1)

df_entrenamiento["monto_compra_boxcox"] = transformador_boxcox.transform(df_entrenamiento[["monto_compra"]] + 1)
df_prueba["monto_compra_boxcox"] = transformador_boxcox.transform(df_prueba[["monto_compra"]] + 1)

In [32]:
transformador_boxcox = PowerTransformer(method="box-cox")
transformador_boxcox.fit(df_entrenamiento[["monto_compra"]] + 1)

df_entrenamiento["monto_compra_boxcox"] = transformador_boxcox.transform(df_entrenamiento[["monto_compra"]] + 1)
df_prueba["monto_compra_boxcox"] = transformador_boxcox.transform(df_prueba[["monto_compra"]] + 1)

In [33]:
df_entrenamiento["monto_compra_boxcox"].skew()

np.float64(-0.5143491146078535)

## Por qué ajusté Box-Cox solo con entrenamiento

Box-Cox aprende un parámetro (lambda) a partir de los datos que ve al hacer `fit()`. Si lo ajustara con el dataset completo, la información del conjunto de prueba se filtraría dentro del propio parámetro, y sería el mismo error de fuga de datos que vimos con el escalado y con el target encoding. Por eso `fit()` solo va con entrenamiento, y el `transform()` se aplica a ambos con ese mismo ajuste.

## Comparación de sesgo antes y después

| Variable | Sesgo (skew) |
|---|---|
| `monto_compra` (original) | sesgo alto positivo |
| `monto_compra_log` | mucho más cercano a 0 |
| `monto_compra_boxcox` | el más cercano a 0 |

Box-Cox fue la transformación que más redujo el sesgo, porque busca automáticamente el parámetro que mejor se ajusta a los datos de entrenamiento, mientras que log1p usa una fórmula fija.

In [34]:
df_final = pd.concat([df_entrenamiento, df_prueba])

In [35]:
df_final.to_csv("S10_PD_Camacho_Serpa_DatasetTransformado.csv", index=False)